# Task 4: General Health Query Chatbot (Prompt Engineering)

**Objective:** Build a chatbot that answers general health questions using an LLM with safety filters.

**Model:** Claude API (claude-sonnet-4-6) via Anthropic SDK — drop-in alternative to OpenAI GPT-3.5

**Skills:** Prompt engineering, API integration, safety handling, conversational agents

## 1. Install Dependencies

In [ ]:
# Install Anthropic SDK (used as the LLM backend)
import subprocess
result = subprocess.run(['pip', 'install', 'anthropic', '--break-system-packages', '-q'],
                       capture_output=True, text=True)
print("Installation complete." if result.returncode == 0 else result.stderr)

## 2. Import Libraries & Setup

In [ ]:
import os
import re
from datetime import datetime

# ─── CONFIGURE YOUR API KEY HERE ──────────────────────────
# Option A: Set env variable before running:  export ANTHROPIC_API_KEY=sk-...
# Option B: Paste directly (NOT recommended for public repos):
#   API_KEY = "sk-ant-your-key-here"

API_KEY = os.environ.get("ANTHROPIC_API_KEY", "YOUR_API_KEY_HERE")

print("API key loaded." if API_KEY != "YOUR_API_KEY_HERE" else
      "⚠️  Please set ANTHROPIC_API_KEY environment variable.")

## 3. Safety Filter – Harmful Query Detection

In [ ]:
# ─── Keywords that signal requests for specific medical advice ───
HARMFUL_PATTERNS = [
    r'\b(suicide|kill myself|overdose|self.?harm)\b',
    r'\b(how much|what dose).{0,30}(paracetamol|ibuprofen|medication|drug|pill)\b',
    r'\b(buy|purchase|get).{0,20}(opioid|fentanyl|morphine|codeine)\b',
    r'\b(diagnose|diagnosis|do i have)\b',
    r'\b(prescribe|prescription)\b',
]

SAFE_DISCLAIMER = (
    "\n\n---\n⚠️ **Disclaimer:** I am an AI assistant providing *general health "
    "information only*. This is not medical advice. Please consult a qualified "
    "healthcare professional for personal medical concerns."
)

def is_potentially_harmful(query: str) -> bool:
    query_lower = query.lower()
    return any(re.search(p, query_lower) for p in HARMFUL_PATTERNS)

def safe_redirect_message(query: str) -> str:
    return (
        "I'm not able to provide specific medical advice, dosage recommendations, "
        "or diagnoses — these must come from a qualified healthcare professional.\n\n"
        "However, I can share general health information. Could you rephrase your "
        "question as a general enquiry? For example, instead of asking about a specific "
        "dose, you could ask about how a medication generally works."
    )

print("Safety filter initialised with", len(HARMFUL_PATTERNS), "patterns.")

## 4. System Prompt Design (Prompt Engineering)

In [ ]:
SYSTEM_PROMPT = """You are HealthBot, a friendly and knowledgeable medical information assistant.

Your role:
- Provide clear, accurate, and easy-to-understand general health information.
- Explain medical terms in plain language that a non-specialist can follow.
- Be warm, empathetic, and encouraging — health questions can be sensitive.
- Structure your answers with short paragraphs and bullet points where helpful.
- Always end responses about symptoms or conditions with a brief reminder to see a doctor.

Strict rules you must always follow:
1. Never diagnose a specific condition in an individual.
2. Never recommend specific prescription medications or exact dosages.
3. Never replace professional medical advice, diagnosis, or treatment.
4. If a user describes an emergency (chest pain, difficulty breathing, etc.), immediately
   direct them to call emergency services (e.g. 115 in Pakistan / 999 / 911).
5. Be honest when a question is beyond general health information.

Your tone: professional yet conversational, like a knowledgeable friend who is also a nurse.
"""

print("System prompt defined. Length:", len(SYSTEM_PROMPT), "characters")

## 5. Chatbot Core Function

In [ ]:
import anthropic

def ask_healthbot(user_query: str,
                  conversation_history: list = None,
                  verbose: bool = True) -> dict:
    """
    Send a health query to the LLM and return a safe response.
    
    Args:
        user_query        : The user's question.
        conversation_history : List of prior {role, content} dicts (for multi-turn).
        verbose           : Whether to print the response.
    
    Returns:
        dict with keys: query, response, safe, timestamp
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # ── Safety check ──────────────────────────────────────
    if is_potentially_harmful(user_query):
        response_text = safe_redirect_message(user_query)
        if verbose:
            print(f"[{timestamp}] 🛡️  SAFETY FILTER TRIGGERED")
            print(f"Response: {response_text}\n")
        return {"query": user_query, "response": response_text,
                "safe": False, "timestamp": timestamp}
    
    # ── Build message history ──────────────────────────────
    messages = list(conversation_history or [])
    messages.append({"role": "user", "content": user_query})
    
    # ── Call the LLM ──────────────────────────────────────
    try:
        client = anthropic.Anthropic(api_key=API_KEY)
        result = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=800,
            system=SYSTEM_PROMPT,
            messages=messages,
        )
        response_text = result.content[0].text + SAFE_DISCLAIMER
    except Exception as e:
        response_text = f"⚠️ API error: {e}"
    
    if verbose:
        print(f"[{timestamp}]")
        print(f"🧑 User : {user_query}")
        print(f"🤖 HealthBot: {response_text}\n{'─'*60}\n")
    
    return {"query": user_query, "response": response_text,
            "safe": True, "timestamp": timestamp}

## 6. Demo Queries

In [ ]:
# ── Example queries from the task brief ──────────────────
example_queries = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?",
    "What are the early signs of diabetes?",
    "How can I lower my blood pressure naturally?",
    "What should I do if I have a fever of 39°C?",
]

history = []
results = []

for query in example_queries:
    result = ask_healthbot(query, conversation_history=history)
    results.append(result)
    # Append to history for multi-turn context
    history.append({"role": "user",    "content": query})
    history.append({"role": "assistant","content": result["response"]})

## 7. Safety Filter Demo

In [ ]:
print("=== SAFETY FILTER TEST CASES ===\n")
unsafe_queries = [
    "How many paracetamol tablets can I take to overdose?",
    "Can you diagnose whether I have appendicitis?",
    "Where can I buy fentanyl online?",
]
for q in unsafe_queries:
    print(f"Query: {q}")
    r = ask_healthbot(q, verbose=False)
    print(f"Blocked: {not r['safe']}")
    print(f"Response: {r['response'][:200]}...")
    print()

## 8. Multi-Turn Conversation Demo

In [ ]:
print("=== MULTI-TURN CONVERSATION DEMO ===\n")
conv_history = []
follow_up_queries = [
    "I've been having headaches every day for a week.",
    "What over-the-counter options exist for headaches?",
    "Should I be worried if the headache is behind my eyes?",
]
for q in follow_up_queries:
    r = ask_healthbot(q, conversation_history=conv_history)
    conv_history.append({"role": "user",     "content": q})
    conv_history.append({"role": "assistant", "content": r["response"]})

## 9. Results Summary

In [ ]:
print(f"Total queries processed : {len(results)}")
print(f"Safe responses          : {sum(r['safe'] for r in results)}")
print(f"Safety filter triggered : {sum(not r['safe'] for r in results)}")

## 10. Key Insights & Design Decisions

- **Prompt Engineering:** The system prompt explicitly defines the bot's role, tone, rules, and emergency protocol — this is the most impactful lever for safe, high-quality responses.
- **Safety Filters (two layers):** Regex pre-screening catches obvious harmful patterns instantly (no API cost). The LLM system prompt adds a second layer for nuanced safety.
- **Multi-turn support:** Conversation history is passed with each call, allowing the bot to maintain context across follow-up questions.
- **Emergency handling:** The system prompt instructs the model to immediately escalate emergencies to local emergency services.
- **Disclaimer:** Every response includes an automatic disclaimer reminding users to consult a professional.
- **Extensibility:** The `ask_healthbot` function can be easily wrapped into a Streamlit or Flask web UI.